Instructions for use on google colab:
1. Run all cells
2. Click link 

In [ ]:
!pip install dash fredapi

In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
from fredapi import Fred
from dash import Dash, dcc, html, Input, Output

In [3]:
fred = Fred(api_key='4f2097fffd781e31f4f401937d28fd23')

In [4]:
series_dict = {
    "Job Openings (Total Nonfarm)": "JTSJOL",
    "Hires (Total Nonfarm)": "JTSHIL",
    "Quits (Total Nonfarm)": "JTSQUL",
    "Layoffs & Discharges (Total Nonfarm)": "JTSLDL",
    "Total Separations (Total Nonfarm)": "JTSTSL",
    "Unemployment Rate": "UNRATE",
    "Labor Force Participation Rate": "CIVPART",
    "Employment Level": "CE16OV",
    "Unemployment Level": "UNEMPLOY",
    "Job Openings in South Census Region": "JTS00SOJOL",
    "Job Openings in Midwest Census Region": "JTS00MWJOL",
    "Job Openings in West Census Region": "JTS00WEJOL",
    "Job Openings in Northeast Census Region": "JTS00NEJOL",
    "Hires in South Census Region": "JTS00SOHIL",
    "Hires in Midwest Census Region": "JTS00MWHIL",
    "Hires in West Census Region": "JTS00WEHIL",
    "Hires in Northeast Census Region": "JTS00NEHIL",
    "Total Separations in South Census Region": "JTS00SOTSL",
    "Total Separations in Midwest Census Region": "JTS00MWTSL",
    "Total Separations in West Census Region": "JTS00WETSL",
    "Total Separations in Northeast Census Region": "JTS00NETSL",
}

start_date = "2001-01-01"
jobs = None
for key, value in series_dict.items():
    temp = (fred.get_series(value, observation_start=start_date)
                .rename(key).reset_index()
                .rename(columns={"index": "DateTime"}))
    temp["DateTime"] = pd.to_datetime(temp["DateTime"])
    jobs = temp if jobs is None else jobs.merge(temp, on="DateTime", how="outer")

jobs = jobs.sort_values("DateTime").reset_index(drop=True)
jobs["Date"] = jobs["DateTime"].dt.date
jobs["Year"] = jobs["DateTime"].dt.year
jobs["Month"] = jobs["DateTime"].dt.month
jobs["date_num"] = jobs["DateTime"].astype("int64")


In [5]:
levels = ["Job Openings (Total Nonfarm)", "Hires (Total Nonfarm)", "Quits (Total Nonfarm)",
          "Layoffs & Discharges (Total Nonfarm)", "Total Separations (Total Nonfarm)",
          "Employment Level", "Unemployment Level"]
scatter_choices = levels + ["Unemployment Rate", "Labor Force Participation Rate"]
openings = ["Job Openings in South Census Region", "Job Openings in Midwest Census Region",
            "Job Openings in West Census Region", "Job Openings in Northeast Census Region"]
hires = ["Hires in South Census Region", "Hires in Midwest Census Region",
         "Hires in West Census Region", "Hires in Northeast Census Region"]
separations = ["Total Separations in South Census Region", "Total Separations in Midwest Census Region",
               "Total Separations in West Census Region", "Total Separations in Northeast Census Region"]
bar_choices = ["Job Openings", "Hires", "Total Separations"]
bar_map = {"Job Openings": openings, "Hires": hires, "Total Separations": separations}
init_lineplot = ["Job Openings (Total Nonfarm)", "Hires (Total Nonfarm)", "Quits (Total Nonfarm)",
                 "Layoffs & Discharges (Total Nonfarm)", "Total Separations (Total Nonfarm)"]

In [6]:
def create_lineplot(data, var_names, title=None, size=[700, 450]):
    fig = px.line(data, "Date", var_names, title=title or "Labor Force Levels Over Time",
                  width=size[0], height=size[1])
    fig.update_layout(yaxis_title="Value x 1000", margin=dict(t=40, b=40, l=40, r=40),
                      legend=dict(title="Variable", orientation="h", yanchor="top", y=-0.15))
    return fig

def create_scatterplot(data, x_var, y_var, title=None, size=[700, 450]):
    tick_vals = np.linspace(data["date_num"].min(), data["date_num"].max(), 6)
    tick_text = pd.to_datetime(tick_vals).strftime("%Y-%m")
    fig = px.scatter(data, x_var, y_var, color="date_num", color_continuous_scale="Viridis",
                     title=title or f"{y_var} vs. {x_var}", width=size[0], height=size[1],
                     hover_data={"Date": True, "date_num": False})
    fig.update_layout(margin=dict(t=40, b=40, l=40, r=40))
    fig.update_coloraxes(colorbar_title="Date", colorbar_tickvals=tick_vals,
                         colorbar_ticktext=tick_text)
    return fig

def create_stackedbarchart(data, var_names, title=None, size=[700, 450]):
    annual = data[["Year"] + var_names].groupby("Year", as_index=False).sum()
    fig = px.bar(annual, x="Year", y=var_names, width=size[0], height=size[1],
                 title=title or "Annual Regional Labor Data")
    fig.update_layout(yaxis_title="Value x 1000", margin=dict(t=40, b=40, l=40, r=40),
                      legend=dict(title="Value", orientation="h", yanchor="top", y=-0.15))
    return fig

def create_heatmap(data, variable, title=None, size=[700, 450]):
    pivot = data.pivot_table(index="Year", columns="Month", values=variable)
    fig = px.imshow(pivot, color_continuous_scale="Viridis", width=size[0], height=size[1],
                    title=title or f"{variable} Heat Map: Year x Month")
    return fig

In [7]:
app = Dash(__name__)

controls = html.Div([
    html.Div([html.B("Years"),
              dcc.RangeSlider(id="year-range", min=2001, max=2026, step=1, value=[2001, 2026],
                              marks={y: str(y) for y in range(2001, 2027, 5)})],
             style={"flex": "1 1 100%", "padding": "8px"}),
    html.Div([html.B("Line Plot Series"),
              dcc.Dropdown(id="line-series", options=levels, value=init_lineplot, multi=True)],
             style={"flex": "1 1 320px", "padding": "8px"}),
    html.Div([html.B("Scatter X"),
              dcc.Dropdown(id="scatter-x", options=scatter_choices, value="Job Openings (Total Nonfarm)")],
             style={"flex": "1 1 220px", "padding": "8px"}),
    html.Div([html.B("Scatter Y"),
              dcc.Dropdown(id="scatter-y", options=scatter_choices, value="Unemployment Level")],
             style={"flex": "1 1 220px", "padding": "8px"}),
    html.Div([html.B("Bar Measure"),
              dcc.Dropdown(id="bar-choice", options=bar_choices, value="Job Openings")],
             style={"flex": "1 1 220px", "padding": "8px"}),
    html.Div([html.B("Heatmap Variable"),
              dcc.Dropdown(id="heatmap-choice", options=scatter_choices, value="Unemployment Rate")],
             style={"flex": "1 1 220px", "padding": "8px"}),
], style={"display": "flex", "flexFlow": "row wrap", "alignItems": "flex-start"})

grid = html.Div([
    dcc.Graph(id="line-graph"),
    dcc.Graph(id="scatter-graph"),
    dcc.Graph(id="bar-graph"),
    dcc.Graph(id="heatmap-graph"),
], style={"display": "grid", "gridTemplateColumns": "1fr 1fr",
          "gridTemplateRows": "auto auto", "gap": "10px"})

app.layout = html.Div([
    html.H1("U.S. Labor Market Dashboard", style={"marginBottom": "0"}),
    html.P("Source: BLS JOLTS via FRED (Federal Reserve Bank of St. Louis)",
           style={"color": "#666", "marginTop": "4px"}),
    controls, grid,
], style={"maxWidth": "1500px", "margin": "0 auto", "fontFamily": "sans-serif"})

@app.callback(
    Output("line-graph", "figure"),
    Output("scatter-graph", "figure"),
    Output("bar-graph", "figure"),
    Output("heatmap-graph", "figure"),
    Input("year-range", "value"),
    Input("line-series", "value"),
    Input("scatter-x", "value"),
    Input("scatter-y", "value"),
    Input("bar-choice", "value"),
    Input("heatmap-choice", "value"),
)
def update_all(year_range, line_series, sx, sy, bar_measure, heatmap_var):
    y0, y1 = year_range
    df = jobs[(jobs["Year"] >= y0) & (jobs["Year"] <= y1)].copy()
    line_fig = create_lineplot(df, list(line_series)) if line_series \
        else px.line(title="Select at least one series for the line plot")
    scatter_fig = create_scatterplot(df, sx, sy)
    bar_fig = create_stackedbarchart(df, bar_map[bar_measure],
                                     title=f"Annual {bar_measure} by Census Region")
    heatmap_fig = create_heatmap(df, heatmap_var)
    return line_fig, scatter_fig, bar_fig, heatmap_fig

In [8]:
app.run(jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/
